# Node Classification of MSD-style Unseen Data

The goal is to classify one or more unseen nodes based on its **node label**. The node label is categorical and is one of [0,1,2,3,4,5,6,7,8]
The model is trained on the MSD dataset.

## What this notebook assumes

You should have already prepared dataset directory containing:

- `graphs.csv`
- `nodes.csv`
- `edges.csv`

### IMPORTANT: Make sure you run S06-15B notebook to assign node labels for each node in nodes.csv.
### The goal is to see if the pre-trained model can predict the node types correctly.
### Labels:

## Room labels and zoning assumptions

The nine room types follow the Modified Swiss Dwellings / CVAAD challenge class order:

| `room_type` | `label` |
|---|---:|
| `bedroom` | `0` |
| `livingroom` | `1` |
| `kitchen` | `2` |
| `dining` | `3` |
| `corridor` | `4` |
| `stairs` | `5` |
| `storeroom` | `6` |
| `bathroom` | `7` |
| `balcony` | `8` |

The four zoning classes are derived from the room type:

| Zoning class | One-hot index | Room types |
|---|---:|---|
| Private/static | `0` | `bedroom` |
| Living/dynamic | `1` | `livingroom`, `kitchen`, `dining`, `corridor` |
| Service/functional | `2` | `stairs`, `storeroom`, `bathroom` |
| Outdoor/semi-outdoor | `3` | `balcony` |

1. Loads the prepared dataset
2. Loads a pre-trained model (given to you in notebooks/Support Files)
3. Predicts the nodes of the test graphs
4. Visualizes the results


## 1. Imports

We use standard Python tools for file handling and tables, and `topologicpy.PyG` for the graph machine learning workflow.

In [2]:
from pathlib import Path
import pandas as pd

from topologicpy.PyG import PyG
from topologicpy.Helper import Helper

## 2. Check the TopologicPy Version

In [ ]:
print("This tutorial requires topologicpy version 0.9.43 or newer.")
print(Helper.Version())

This tutorial requires topologicpy version 0.9.31 or newer.
The version that you are using (0.9.43) is EQUAL TO the latest version available on PyPI.


## 3. Set your renderer:
* Visual studio code: "vscode"
* Google Colab: "colab"
* Browser: "browser"

In [4]:
renderer = "vscode"

In [5]:
# ------------------------------------------------------------------
# DATASET LOCATION
# ------------------------------------------------------------------
# Change this path if your prepared clean dataset is elsewhere.
# The folder must contain graphs.csv, nodes.csv, and edges.csv.
DATASET_PATH = Path.cwd().parent / '03_node_classification' / 'results'
MODEL_PATH = Path.cwd().parent / '03_node_classification' / 'assets' / 'msd_node_classifier.pt'
VIS_PATH = Path.cwd().parent / '03_node_classification' / 'results'

# ------------------------------------------------------------------
# TASK DEFINITION
# ------------------------------------------------------------------
PREDICTION_LEVEL = "node"               # Prediction target level: graph | node | edge | link.
TASK = "classification"                 # Learning task type: classification | regression | link_prediction.
GRAPH_LABEL_TYPE = "categorical"        # Type of graph-level label: continuous for regression targets such as cooling/heating load.
NODE_LABEL_TYPE = "categorical"         # Type of node-level label: categorical class index, e.g. room type or element type.
EDGE_LABEL_TYPE = "categorical"         # Type of edge-level label: categorical class index; currently not used in this setup.



## 5. Check the dataset files

Before loading the dataset into PyG, it is good practice to verify that the three required CSV files exist.

In [6]:
required_files = [
    DATASET_PATH / "graphs.csv",
    DATASET_PATH / "nodes.csv",
    DATASET_PATH / "edges.csv",
]

for f in required_files:
    print(f"{f.name}: {'FOUND' if f.exists() else 'MISSING'} -> {f}")

if not all(f.exists() for f in required_files):
    raise FileNotFoundError(
        "One or more required CSV files are missing. "
        "Check PREPARED_DATASET_DIR before proceeding."
    )

graphs.csv: FOUND -> C:\Users\sarwj\OneDrive - Cardiff University\IAAC\2025-26\S3 - Buildings As Graphs\notebooks\Supporting Files\dataset_node_classification\graphs.csv
nodes.csv: FOUND -> C:\Users\sarwj\OneDrive - Cardiff University\IAAC\2025-26\S3 - Buildings As Graphs\notebooks\Supporting Files\dataset_node_classification\nodes.csv
edges.csv: FOUND -> C:\Users\sarwj\OneDrive - Cardiff University\IAAC\2025-26\S3 - Buildings As Graphs\notebooks\Supporting Files\dataset_node_classification\edges.csv


## 6. Inspect the CSV schema

This step is pedagogically useful because it makes the node-classification setup visible.

For node classification, the important pieces are:

### `nodes.csv`
- node identifiers
- node labels
- node features
- train/validation/test masks

### `edges.csv`
- source node
- destination node
- optional edge features

### `graphs.csv`
- graph identifier
- number of nodes

In [7]:
graphs_df = pd.read_csv(DATASET_PATH / "graphs.csv")
nodes_df = pd.read_csv(DATASET_PATH / "nodes.csv")
edges_df = pd.read_csv(DATASET_PATH / "edges.csv")

print("graphs.csv shape:", graphs_df.shape)
print("nodes.csv shape:", nodes_df.shape)
print("edges.csv shape:", edges_df.shape)

graphs.csv shape: (500, 2)
nodes.csv shape: (15914, 13)
edges.csv shape: (32154, 6)


In [8]:
print("graphs.csv columns:")
print(list(graphs_df.columns))
print()

print("nodes.csv columns:")
print(list(nodes_df.columns))
print()

print("edges.csv columns:")
print(list(edges_df.columns))

graphs.csv columns:
['graph_id', 'num_nodes']

nodes.csv columns:
['graph_id', 'node_id', 'label', 'feat_zoning_type_0', 'feat_zoning_type_1', 'feat_zoning_type_2', 'feat_zoning_type_3', 'feat_connectivity_0', 'feat_connectivity_1', 'feat_connectivity_2', 'train_mask', 'val_mask', 'test_mask']

edges.csv columns:
['graph_id', 'src_id', 'dst_id', 'feat_connectivity_0', 'feat_connectivity_1', 'feat_connectivity_2']


## 7. Load the dataset (yours)

In [9]:
pyg_2 = PyG.ByCSVPath(
    path=str(DATASET_PATH),
    level=PREDICTION_LEVEL,
    task=TASK,
    graphLabelType=GRAPH_LABEL_TYPE,
    nodeLabelType=NODE_LABEL_TYPE,
    edgeLabelType=EDGE_LABEL_TYPE
)

## 8. Load the pre-trained model

In [10]:
pyg_2.LoadModel(str(MODEL_PATH))

## 2. Predict node labels for all nodes

This step attaches predictions to the internal PyG data objects.

It is useful when you want to inspect which nodes were correctly or incorrectly classified.

In [11]:
_ = pyg_2.Predict(split="all", return_probs=True, attach_to_data=True)
print("Predictions attached to the dataset.")

Predictions attached to the dataset.


## 3. Export node-level predictions to CSV

The exported table is useful for:

- error analysis
- joining predictions back to geometry
- reviewing results graph by graph

In [13]:
from pathlib import Path
import pandas as pd
import numpy as np

def _to_class_index(value):
    """
    Convert scalar / length-1 array / one-hot vector to an integer class index.
    """
    arr = np.asarray(value)
    arr = np.squeeze(arr)

    if arr.ndim == 0:
        return int(arr)

    if arr.ndim == 1:
        if arr.size == 1:
            return int(arr[0])
        return int(np.argmax(arr))

    raise ValueError(f"Cannot convert value with shape {arr.shape} to class index.")

def export_node_predictions(pyg_obj, output_csv: Path) -> pd.DataFrame:
    pred_report = pyg_obj.Predict(
        split="all",
        return_probs=True,
        attach_to_data=True
    )

    pred_by_graph = pred_report["pred"]
    y_true_by_graph = pred_report["y_true"]
    prob_by_graph = pred_report.get("prob", None)

    print("Number of graphs in pred_report['pred']:", len(pred_by_graph))
    print("Number of graphs in pyg_obj.data_list:", len(pyg_obj.data_list))

    rows = []

    for graph_idx, data in enumerate(pyg_obj.data_list):
        graph_id = int(data.graph_id.item()) if hasattr(data, "graph_id") else graph_idx
        n = data.num_nodes

        graph_pred = pred_by_graph[graph_idx]
        graph_true = y_true_by_graph[graph_idx]
        graph_prob = prob_by_graph[graph_idx] if prob_by_graph is not None else None

        graph_pred = np.asarray(graph_pred)
        graph_true = np.asarray(graph_true)
        if graph_prob is not None:
            graph_prob = np.asarray(graph_prob)

        print(
            f"graph {graph_idx}: "
            f"num_nodes={n}, pred_shape={graph_pred.shape}, true_shape={graph_true.shape}, "
            f"prob_shape={None if graph_prob is None else graph_prob.shape}"
        )

        train_mask = data.train_mask.detach().cpu().numpy() if hasattr(data, "train_mask") else None
        val_mask = data.val_mask.detach().cpu().numpy() if hasattr(data, "val_mask") else None
        test_mask = data.test_mask.detach().cpu().numpy() if hasattr(data, "test_mask") else None

        if len(graph_pred) != n:
            raise ValueError(
                f"Prediction length mismatch in graph {graph_idx}: "
                f"len(graph_pred)={len(graph_pred)} but num_nodes={n}"
            )
        if len(graph_true) != n:
            raise ValueError(
                f"Ground-truth length mismatch in graph {graph_idx}: "
                f"len(graph_true)={len(graph_true)} but num_nodes={n}"
            )

        for node_idx in range(n):
            y_true_i = _to_class_index(graph_true[node_idx])
            y_pred_i = _to_class_index(graph_pred[node_idx])

            row = {
                "graph_id": graph_id,
                "node_id": node_idx,
                "y_true": y_true_i,
                "y_pred": y_pred_i,
            }

            if graph_prob is not None:
                prob_i = np.asarray(graph_prob[node_idx]).squeeze()
                if prob_i.ndim == 1 and y_pred_i < prob_i.size:
                    row["y_pred_prob"] = float(prob_i[y_pred_i])

            if train_mask is not None:
                row["train_mask"] = bool(train_mask[node_idx])
            if val_mask is not None:
                row["val_mask"] = bool(val_mask[node_idx])
            if test_mask is not None:
                row["test_mask"] = bool(test_mask[node_idx])

            rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)
    return df

predictions_csv = DATASET_PATH / "node_predictions_baseline.csv"
predictions_df = export_node_predictions(pyg_2, predictions_csv)

print(f"Saved predictions to: {predictions_csv}")
display(predictions_df.head())

Number of graphs in pred_report['pred']: 500
Number of graphs in pyg_obj.data_list: 500
graph 0: num_nodes=19, pred_shape=(19,), true_shape=(19,), prob_shape=(19, 9)
graph 1: num_nodes=16, pred_shape=(16,), true_shape=(16,), prob_shape=(16, 9)
graph 2: num_nodes=16, pred_shape=(16,), true_shape=(16,), prob_shape=(16, 9)
graph 3: num_nodes=52, pred_shape=(52,), true_shape=(52,), prob_shape=(52, 9)
graph 4: num_nodes=28, pred_shape=(28,), true_shape=(28,), prob_shape=(28, 9)
graph 5: num_nodes=28, pred_shape=(28,), true_shape=(28,), prob_shape=(28, 9)
graph 6: num_nodes=42, pred_shape=(42,), true_shape=(42,), prob_shape=(42, 9)
graph 7: num_nodes=24, pred_shape=(24,), true_shape=(24,), prob_shape=(24, 9)
graph 8: num_nodes=23, pred_shape=(23,), true_shape=(23,), prob_shape=(23, 9)
graph 9: num_nodes=26, pred_shape=(26,), true_shape=(26,), prob_shape=(26, 9)
graph 10: num_nodes=16, pred_shape=(16,), true_shape=(16,), prob_shape=(16, 9)
graph 11: num_nodes=38, pred_shape=(38,), true_shape=

,graph_id,node_id,y_true,y_pred,y_pred_prob,train_mask,val_mask,test_mask
0,0,0,0,0,1.000000,True,False,False
1,0,1,0,0,1.000000,True,False,False
2,0,2,7,7,0.914835,True,False,False
3,0,3,1,4,0.635013,True,False,False
4,0,4,7,7,0.914835,True,False,False


## 16. Compare Prediction vs. True visually

In [ ]:
from topologicpy.Graph import Graph

graphs = Graph.ByCSVPath(path=str(VIS_PATH), nodeFeaturesKeys=['true', 'pred'])
print(f'Graphs loaded: {len(graphs)}')

In [ ]:
import pandas as pd

nodes_df = pd.read_csv(DATASET_PATH / 'nodes.csv')
preds_df = pd.read_csv(DATASET_PATH / 'node_predictions_baseline.csv')

nodes_df = nodes_df.merge(
    preds_df[['graph_id', 'node_id', 'y_true', 'y_pred']],
    on=['graph_id', 'node_id'],
    how='left'
).rename(columns={'y_true': 'true', 'y_pred': 'pred'})

nodes_df.to_csv(DATASET_PATH / 'nodes.csv', index=False)
print('Predictions merged into nodes.csv')

In [ ]:
from topologicpy.Dictionary import Dictionary
from topologicpy.Topology import Topology
from topologicpy.Color import Color

g = graphs[0]  # Only one graph — Type K

verts = Graph.Vertices(g)
for v in verts:
    d = Topology.Dictionary(v)
    true = int(Dictionary.ValueAtKey(d, "true"))
    pred = int(Dictionary.ValueAtKey(d, "pred"))
    if not true == pred:
        size = 30
        true_color = "red"
        pred_color = "red"
    else:
        size = 14
        true_color = Color.ByValueInRange(true, minValue=0, maxValue=8)
        pred_color = Color.ByValueInRange(pred, minValue=0, maxValue=8)
    d = Dictionary.SetValuesAtKeys(d, ["true_color", "pred_color", "size", "true", "pred"], [true_color, pred_color, size, true, pred])
    v = Topology.SetDictionary(v, d)

g = Graph.Reshape(g)
Topology.Show(g, vertexSize=6, vertexSizeKey="size", vertexColorKey="true_color", showVertexLabel=True, vertexLabelKey="true", backgroundColor="white", camera=[0,0,3], vertexLabelFontSize=18)
Topology.Show(g, vertexSize=6, vertexSizeKey="size", vertexColorKey="pred_color", showVertexLabel=True, vertexLabelKey="pred", backgroundColor="white", camera=[0,0,3], vertexLabelFontSize=18)